# 01 · Cohort Selection — PLAN_end_to_end PART A

Rebuilds the cohort from scratch on **MIMIC-CXR + MIMIC-IV via BigQuery**. The old
cohort is abandoned: it merged micro on `subject_id` with **no time anchor**
(mean 94.85 micro rows/study, max 1427 — a Cartesian product) and leaked the
future (`frac_studies_with_future_organism = 0.448`).

**Two governing constraints (§Intro):**
1. **Causality** — every EHR feature must have been available to a clinician *at
   the moment the film was read*.
2. **No selection on EHR** — inclusion never depends on EHR availability or label
   values. The ~50% of studies with no prior micro are **data**, not missingness.

This notebook covers **A0–A2 + labels + split** and emits `cohort_base.parquet`.
Micro tiers and labs (A3–A7) live in `02_ehr_features.ipynb` — deliberately
separate, because that is where the leakage asserts (§E1) must fail fast.

> ### ⚠️ Run order is **03 → 01 → 02**, not 01 → 02 → 03
> The file numbers are not the run order. `03_view_resolution.ipynb` has no
> dependency on this notebook, but this notebook depends on **its** output:
> `ViewPosition` is blank on 4.2% of images, and the inclusion rule below counts
> **recovered** frontal/lateral views (§A1 step 3). Running 01 first raises
> `FileNotFoundError` on `view_per_study.parquet` rather than silently building the
> smaller, pre-recovery cohort.

---
### Deviations from PLAN (confirmed with Tuan — plan text says otherwise)

| Item | PLAN says | We do | Why |
|---|---|---|---|
| Label count | 14 CheXpert (`labels_0..13`, `y[14]`) | **13 — drop `Support Devices`** | not a pathology (tubes/lines/pacemakers) |
| U-policy | "confirm U-Ignore" (open question #4) | **U-Ones** (`-1 → 1`) | favours recall on rare labels; no loss mask needed |
| Outpatient | open question #1 | **keep** | dropping selects on encounter type |


## C0 · Config

Every knob in one place. Nothing about the cohort depends on EHR presence.

In [ ]:
import os, re, numpy as np, pandas as pd

PROJECT = 'mimic-ehr-v310'
HOSP    = 'physionet-data.mimiciv_3_1_hosp'
CXR     = 'physionet-data.mimic_cxr_jpg'
ED      = None                      # resolved in C1 (do NOT guess the name)

try:
    from google.colab import auth, drive
    auth.authenticate_user(); drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
from google.cloud import bigquery
client = bigquery.Client(project=PROJECT, location='US')

OUT_DIR = os.path.join(DATA_DIR, 'cohort_v2')
os.makedirs(OUT_DIR, exist_ok=True)

# --- decisions (see deviations table above) ---
DROP_LABELS  = ['Support Devices']       # -> 13 labels
U_POLICY     = 'ones'                    # {'ones','zeros','ignore'}
KEEP_OUTPATIENT = True
SPLIT_SOURCE = 'regenerate'              # {'regenerate','official'}  <- see C6
SPLIT_SEED, SPLIT_FRAC = 42, (0.70, 0.15, 0.15)

print('OUT_DIR =', OUT_DIR)

## C1 · VERIFY names before using them (§B1 ethos: *do not assume*)

Resolves the ED dataset id and the CheXpert label column names from the actual
schema, and **fails loudly** rather than guessing. Every wrong assumption so far
in this project came from skipping this step.

In [ ]:
# --- ED dataset: find it, don't guess ---
all_ds = [d.dataset_id for d in client.list_datasets('physionet-data')]
cands  = [d for d in all_ds if re.search(r'mimic.*(_ed$|_ed_|iv_ed)', d, re.I)]
print('mimic datasets:', [d for d in all_ds if 'mimic' in d.lower()])
if not cands:
    raise RuntimeError('No MIMIC-ED dataset found. §A2 needs edstays for setting="ed".')
ED = f'physionet-data.{sorted(cands)[-1]}'
print('ED dataset ->', ED)

# --- CheXpert label columns: read them off the schema ---
chex_tbl  = client.get_table(f'{CXR}.chexpert')
chex_cols = [f.name for f in chex_tbl.schema]
LABEL_COLS_ALL = [c for c in chex_cols if c not in ('subject_id','study_id')]
print(f'\nchexpert: {chex_tbl.num_rows:,} rows, {len(LABEL_COLS_ALL)} label cols')
print(LABEL_COLS_ALL)

def norm(c): return c.replace('_',' ').strip().lower()
drop_norm  = {norm(c) for c in DROP_LABELS}
LABEL_COLS = [c for c in LABEL_COLS_ALL if norm(c) not in drop_norm]
dropped    = [c for c in LABEL_COLS_ALL if norm(c) in drop_norm]
assert dropped, f'DROP_LABELS {DROP_LABELS} matched nothing in {LABEL_COLS_ALL}'
print(f'\ndropped ({len(dropped)}): {dropped}')
print(f'KEEPING {len(LABEL_COLS)} labels: {LABEL_COLS}')
assert len(LABEL_COLS) == 13, f'expected 13 labels, got {len(LABEL_COLS)}'

# --- edstays schema sanity ---
ed_cols = [f.name for f in client.get_table(f'{ED}.edstays').schema]
print('\nedstays cols:', ed_cols)
for need in ('subject_id','stay_id','intime','outtime'):
    assert need in ed_cols, f'edstays missing {need}'

## C2 · A0 study table + A1 flags + A2 encounter assignment

One query, all of PART A's structural work:

- **A0** — one row per `study_id`; `index_time` = `MIN(study_datetime)`; count
  distinct times per study so §A0's assert is checkable.
- **A1** — inclusion *flags* only (filtering happens in C3 so the attrition table
  can be built). Nothing here touches micro/labs/labels.
- **A2** — `cur_hadm` via `admittime <= index_time <= dischtime`, `cur_edstay` via
  `intime <= index_time <= outtime`. Overlaps → latest `admittime`, count kept so
  §A2's assert is checkable. Neither → `outpatient`.

In [ ]:
q = f"""
WITH s AS (
  SELECT study_id,
         ANY_VALUE(subject_id)                                  AS subject_id,
         MIN(CAST(study_datetime AS TIMESTAMP))                 AS index_time,
         COUNT(DISTINCT CAST(study_datetime AS TIMESTAMP))      AS n_distinct_times,
         COUNT(*)                                               AS n_images,
         COUNTIF(ViewPosition IS NOT NULL
                 AND TRIM(ViewPosition) != '')                  AS n_usable_view
  FROM `{CXR}.metadata`
  GROUP BY study_id
),
chex AS (SELECT DISTINCT study_id, TRUE AS has_chexpert FROM `{CXR}.chexpert`),
pat  AS (SELECT DISTINCT subject_id, TRUE AS in_patients  FROM `{HOSP}.patients`),
adm AS (
  SELECT study_id, hadm_id, n_match FROM (
    SELECT s.study_id, a.hadm_id,
           COUNT(*)   OVER (PARTITION BY s.study_id)                        AS n_match,
           ROW_NUMBER() OVER (PARTITION BY s.study_id
                              ORDER BY CAST(a.admittime AS TIMESTAMP) DESC) AS rn
    FROM s JOIN `{HOSP}.admissions` a
      ON a.subject_id = s.subject_id
     AND s.index_time BETWEEN CAST(a.admittime AS TIMESTAMP)
                          AND CAST(a.dischtime AS TIMESTAMP)
  ) WHERE rn = 1
),
eds AS (
  SELECT study_id, stay_id, n_match_ed FROM (
    SELECT s.study_id, e.stay_id,
           COUNT(*)   OVER (PARTITION BY s.study_id)                     AS n_match_ed,
           ROW_NUMBER() OVER (PARTITION BY s.study_id
                              ORDER BY CAST(e.intime AS TIMESTAMP) DESC) AS rn
    FROM s JOIN `{ED}.edstays` e
      ON e.subject_id = s.subject_id
     AND s.index_time BETWEEN CAST(e.intime AS TIMESTAMP)
                          AND CAST(e.outtime AS TIMESTAMP)
  ) WHERE rn = 1
)
SELECT s.*,
       COALESCE(chex.has_chexpert, FALSE) AS has_chexpert,
       COALESCE(pat.in_patients,  FALSE)  AS in_patients,
       adm.hadm_id      AS cur_hadm,
       COALESCE(adm.n_match, 0)    AS n_hadm_match,
       eds.stay_id      AS cur_edstay,
       COALESCE(eds.n_match_ed, 0) AS n_edstay_match
FROM s
LEFT JOIN chex USING (study_id)
LEFT JOIN pat  ON pat.subject_id = s.subject_id
LEFT JOIN adm  USING (study_id)
LEFT JOIN eds  USING (study_id)
"""
studies = client.query(q).to_dataframe()
print('studies pulled:', len(studies))

# ---- view comes from 03, not from the raw DICOM field ----
# ViewPosition is blank on 4.2% of images. 03 recovers it (procedure description +
# within-study elimination) and that recovers 2,153 studies which this filter would
# otherwise drop for a metadata reason rather than a clinical one.
VIEW_PS = os.path.join(OUT_DIR, 'view_per_study.parquet')
if not os.path.exists(VIEW_PS):
    raise FileNotFoundError(
        f'{VIEW_PS} not found.\n'
        '    -> RUN 03_view_resolution.ipynb FIRST.\n'
        '       The file numbering is NOT the run order: 03 has no dependency on 01,\n'
        '       but 01 depends on 03. Order is 03 -> 01 -> 02.')
vps = pd.read_parquet(VIEW_PS)[['study_id', 'n_frontal', 'n_lateral']]
studies = studies.merge(vps, on='study_id', how='left')
studies[['n_frontal', 'n_lateral']] = studies[['n_frontal', 'n_lateral']].fillna(0)
studies['n_view_ok'] = studies.n_frontal + studies.n_lateral
print(f'view map merged: {int(studies.n_view_ok.gt(0).sum()):,} studies have >=1 '
      f'frontal/lateral image')
print(f'   raw ViewPosition field alone would give: '
      f'{int(studies.n_usable_view.gt(0).sum()):,}')
print(f'   -> recovered by 03: '
      f'{int((studies.n_view_ok.gt(0) & studies.n_usable_view.eq(0)).sum()):,} studies')

# §A0 assert: one acquisition time per study
multi_t = int((studies['n_distinct_times'] > 1).sum())
print(f'studies with >1 distinct study_datetime: {multi_t} '
      f'({multi_t/len(studies):.3%})  -> took MIN, logged')

# §A2 assert: no study matches >1 hadm
multi_h = int((studies['n_hadm_match'] > 1).sum())
print(f'studies matching >1 hadm_id: {multi_h} ({multi_h/len(studies):.3%}) '
      f' -> took latest admittime, logged')

## C3 · A1 inclusion + CONSORT attrition table

Include: valid `index_time` · has CheXpert · ≥1 usable `ViewPosition` ·
`subject_id` in `hosp.patients`. **Exclude nothing** on micro/lab/label.

In [ ]:
rows, n = [], len(studies)
rows.append(('0. all studies in mimic_cxr_jpg.metadata', n, 0))

keep = studies['index_time'].notna()
rows.append(('1. valid index_time', int(keep.sum()), int((~keep).sum())))

step = keep & studies['has_chexpert']
rows.append(('2. + has CheXpert labels', int(step.sum()), int((keep & ~studies['has_chexpert']).sum())))
keep = step

step = keep & (studies['n_view_ok'] >= 1)
rows.append(('3. + >=1 frontal/lateral view (recovered, see 03)',
             int(step.sum()), int((keep & (studies['n_view_ok'] < 1)).sum())))
keep = step

step = keep & studies['in_patients']
rows.append(('4. + subject_id in hosp.patients', int(step.sum()), int((keep & ~studies['in_patients']).sum())))
keep = step

if not KEEP_OUTPATIENT:
    has_enc = studies['cur_hadm'].notna() | studies['cur_edstay'].notna()
    step = keep & has_enc
    rows.append(('5. + inpatient/ED only', int(step.sum()), int((keep & ~has_enc).sum())))
    keep = step
else:
    rows.append(('5. outpatient KEPT (no EHR-based exclusion)', int(keep.sum()), 0))

attrition = pd.DataFrame(rows, columns=['step', 'n_remaining', 'n_dropped'])
print(attrition.to_string(index=False))

cohort = studies[keep].copy().reset_index(drop=True)
attrition.to_csv(os.path.join(OUT_DIR, 'attrition.csv'), index=False)
print(f'\ncohort: {len(cohort):,} studies / {cohort.subject_id.nunique():,} subjects')

## C4 · A2 `setting`

`inpatient` if `cur_hadm` · else `ed` if `cur_edstay` · else `outpatient`.
Per §A2 this is **both a feature and a stratification variable** — inpatient vs ED
have different labeling regimes and missingness, and §C requires every metric
reported stratified by it. Outpatient dominance is also what explains Tier-1
rarity (≈11.2%) later — that is expected, not a bug.

In [ ]:
cohort['setting'] = np.where(cohort['cur_hadm'].notna(), 'inpatient',
                     np.where(cohort['cur_edstay'].notna(), 'ed', 'outpatient'))
vc = cohort['setting'].value_counts()
print('setting distribution:')
for k, v in vc.items():
    print(f'   {k:<12} {v:>8,}  ({v/len(cohort):6.2%})')

## C5 · Labels — 13, U-Ones

CheXpert encodes `1`=positive, `0`=negative, `-1`=uncertain, blank/NaN=not
mentioned. **U-Ones** (confirmed): `-1 → 1`. NaN → 0 (not mentioned = negative,
the standard CheXpert reading).

In [ ]:
chex = client.query(f"SELECT * FROM `{CXR}.chexpert`").to_dataframe()
print('chexpert rows:', len(chex))

before = chex[LABEL_COLS].apply(lambda s: s.value_counts(dropna=False)).fillna(0).astype(int)
print('\nraw label values BEFORE U-policy (rows = value):')
print(before.to_string())

lab = chex[['study_id'] + LABEL_COLS].copy()
for c in LABEL_COLS:
    v = lab[c]
    if U_POLICY == 'ones':    v = v.replace(-1, 1)
    elif U_POLICY == 'zeros': v = v.replace(-1, 0)
    elif U_POLICY == 'ignore':pass                    # keep -1, loss must mask
    lab[c] = v.fillna(0).astype('int8' if U_POLICY != 'ignore' else 'float32')

cohort = cohort.merge(lab, on='study_id', how='left')      # LEFT: never drop studies
assert cohort[LABEL_COLS].isna().sum().sum() == 0, 'label NaN after merge -> has_chexpert lied'
print(f'\nU_POLICY={U_POLICY} applied. label matrix: {cohort[LABEL_COLS].shape}')

## C6 · Split — regenerated, subject-grouped, **stratified on tail labels**

**Why not the official MIMIC-CXR split.** It is subject-level and reproducible,
but it was designed for the *full* 377k image set. We re-selected the cohort
(13 labels, `ViewPosition` filter, `patients` join), so its balance no longer
holds — and it never balanced rare labels to begin with. With `Pleural Other` /
`Fracture` near the 2% floor, an unstratified split can hand `val`/`test` a
handful of positives and make AUPRC on exactly the labels this research question
is about pure noise.

**Algorithm — iterative stratification (Sechidis et al. 2011), at subject level:**

1. Aggregate labels to `subject_id` (a subject's positive-study count per label).
2. Walk labels **rarest first** — the tail gets first claim on scarce positives.
3. Each subject goes to the split with the largest *remaining need* for that
   label; ties broken by remaining study capacity.
4. Subjects with no positives fill by remaining capacity.

Whole subjects move together, so **disjointness is structural**, not checked-in
after the fact.

> This does **not** violate §A1 ("no selection on EHR or label values"). §A1
> governs *cohort inclusion* — every study stays in. Balancing *which split* a
> subject lands in is standard practice for imbalanced multi-label and changes no
> row's membership.

`hash()` is not used anywhere: Python salts string hashes per process
(`PYTHONHASHSEED`), so `hash(s) % 100` silently yields a **different split on
every run**. The `official` fallback path now uses `hashlib.md5`.

In [ ]:
def stratified_group_split(df, label_cols, fracs=(0.70,0.15,0.15), seed=42):
    """Iterative stratification over subjects; rarest label claims first."""
    splits = ['train','val','test']
    g   = df.groupby('subject_id')[label_cols].sum()          # positives per subject per label
    nst = df.groupby('subject_id').size().reindex(g.index)    # studies per subject
    subs, L, ns = g.index.to_numpy(), g.to_numpy(float), nst.to_numpy(float)

    total_lab = L.sum(0)                                      # positives per label overall
    need  = {s: total_lab * f for s, f in zip(splits, fracs)} # remaining label quota
    cap   = {s: ns.sum()     * f for s, f in zip(splits, fracs)}  # remaining study quota
    rng, out, left = np.random.default_rng(seed), {}, set(range(len(subs)))

    for k in np.argsort(total_lab):                           # rarest label FIRST
        idx = [i for i in left if L[i, k] > 0]
        rng.shuffle(idx)
        for i in idx:
            s = max(splits, key=lambda s: (need[s][k], cap[s]))
            out[subs[i]] = s; need[s] -= L[i]; cap[s] -= ns[i]; left.discard(i)

    rest = list(left); rng.shuffle(rest)                      # subjects with no positives
    for i in rest:
        s = max(splits, key=lambda s: cap[s])
        out[subs[i]] = s; need[s] -= L[i]; cap[s] -= ns[i]
    return out

if SPLIT_SOURCE == 'regenerate':
    m = stratified_group_split(cohort, LABEL_COLS, SPLIT_FRAC, SPLIT_SEED)
    cohort['split'] = cohort['subject_id'].map(m)
else:
    import hashlib
    sp = client.query(f"SELECT DISTINCT study_id, split FROM `{CXR}.split`").to_dataframe()
    sp['split'] = sp['split'].replace({'validate':'val','valid':'val','dev':'val'})
    cohort = cohort.merge(sp.drop_duplicates('study_id'), on='study_id', how='left')
    miss = cohort['split'].isna()
    if miss.any():   # deterministic md5 (NOT hash(): salted per process -> irreproducible)
        print(f'{miss.sum()} studies lack an official split -> deterministic md5 by subject')
        h = cohort.loc[miss,'subject_id'].astype(str).map(
                lambda s: int(hashlib.md5(s.encode()).hexdigest(), 16) % 100)
        cohort.loc[miss,'split'] = np.where(h < 70,'train', np.where(h < 85,'val','test'))

assert cohort['split'].notna().all(), 'unassigned studies'
S = {s:set(cohort.loc[cohort.split==s,'subject_id']) for s in ['train','val','test']}
for a in S:
    for b in S:
        if a < b: assert not (S[a] & S[b]), f'LEAKAGE: {len(S[a]&S[b])} subjects in {a}&{b}'
print(f'split source: {SPLIT_SOURCE}  |  subject-level disjoint OK')
print(cohort.groupby('split').agg(n_studies=('study_id','size'),
                                  n_subjects=('subject_id','nunique')).to_string())

### C6-bis · Did stratification actually work?

Asserting disjointness proves nothing about *balance*. The tail is what matters:
if `n_pos` in `val`/`test` is single-digit for a rare label, its AUPRC is noise no
matter how good the model is. Deviation is measured in **prevalence points vs the
cohort overall**, worst offender reported.

In [ ]:
rows = []
for c in LABEL_COLS:
    o = cohort[c].mean()
    r = {'label': c.replace('_',' '), 'overall_%': round(o*100, 2)}
    for s in ['train','val','test']:
        sub = cohort[cohort.split == s]
        r[f'{s}_%']  = round(sub[c].mean()*100, 2)
        r[f'{s}_n']  = int(sub[c].sum())
    r['max_dev_pp'] = round(max(abs(r[f'{s}_%'] - r['overall_%']) for s in ['train','val','test']), 2)
    rows.append(r)
bal = pd.DataFrame(rows).sort_values('overall_%')
print('per-label prevalence by split (sorted rarest first):')
print(bal.to_string(index=False))

worst = bal['max_dev_pp'].max()
print(f'\nworst deviation from overall prevalence: {worst:.2f} pp')
min_pos = min(bal[['val_n','test_n']].min())
print(f'smallest positive count in val/test: {min_pos}')
if min_pos < 20:
    print('  !! WARNING: a label has <20 positives in val/test -> its AUPRC will be'
          ' unstable regardless of the model. Consider merging it or reporting it'
          ' with a CI rather than a point estimate.')
assert worst < 5.0, f'stratification failed: {worst}pp deviation'
print('\nstratification OK')

## C7 · Emit `cohort_base.parquet` + the `study_id → [dicom_id]` side table

The side table is required by §B2 for embedding pooling and the view ablation.

In [ ]:
dicom = client.query(f"""
  SELECT study_id, dicom_id, subject_id, ViewPosition
  FROM `{CXR}.metadata`
""").to_dataframe()
dicom = dicom[dicom.study_id.isin(set(cohort.study_id))]

base_cols = ['study_id','subject_id','index_time','setting','cur_hadm','cur_edstay',
             'n_images','n_usable_view','n_frontal','n_lateral','n_view_ok',
             'split'] + LABEL_COLS
cohort[base_cols].to_parquet(os.path.join(OUT_DIR,'cohort_base.parquet'), index=False)
dicom.to_parquet(os.path.join(OUT_DIR,'study_dicom_map.parquet'), index=False)

import json as _json
meta = dict(n_studies=int(len(cohort)), n_subjects=int(cohort.subject_id.nunique()),
            n_images=int(len(dicom)), labels=LABEL_COLS, n_labels=len(LABEL_COLS),
            dropped_labels=dropped, u_policy=U_POLICY,
            keep_outpatient=bool(KEEP_OUTPATIENT), split_source=SPLIT_SOURCE)
_json.dump(meta, open(os.path.join(OUT_DIR,'cohort_meta.json'),'w'), indent=2)
for f in sorted(os.listdir(OUT_DIR)):
    print(' ', f, round(os.path.getsize(os.path.join(OUT_DIR,f))/1e6,2),'MB')

## C8 · [YÊU CẦU 1] Số lượng & hình thái dữ liệu

In [ ]:
n_s, n_p, n_i = len(cohort), cohort.subject_id.nunique(), len(dicom)
print('='*64); print('COHORT SHAPE'); print('='*64)
print(f'  studies (rows)      : {n_s:,}')
print(f'  subjects            : {n_p:,}   ({n_s/n_p:.2f} studies/subject)')
print(f'  images (dicom)      : {n_i:,}   ({n_i/n_s:.2f} images/study)')
print(f'  labels              : {len(LABEL_COLS)}  (dropped: {dropped})')
print(f'  table shape         : {cohort[base_cols].shape}')

print('\n--- split ---')
print(cohort.groupby('split').agg(studies=('study_id','size'),
      subjects=('subject_id','nunique')).assign(
      pct=lambda d:(d.studies/n_s*100).round(2)).to_string())

print('\n--- setting (stratification variable, §A2) ---')
print(cohort.groupby('setting').agg(studies=('study_id','size'),
      subjects=('subject_id','nunique')).assign(
      pct=lambda d:(d.studies/n_s*100).round(2)).to_string())

print('\n--- setting x split ---')
print(pd.crosstab(cohort.setting, cohort.split, margins=True).to_string())

print('\n--- images per study ---')
print(cohort['n_images'].describe().round(2).to_string())

print('\n--- index_time range ---')
print(f"  {cohort.index_time.min()}  ->  {cohort.index_time.max()}")

print('\n--- dtypes ---')
print(cohort[base_cols].dtypes.to_string())

print('\n--- labels per study (multi-label density) ---')
k = cohort[LABEL_COLS].sum(axis=1)
print(f'  mean {k.mean():.2f} | median {int(k.median())} | max {int(k.max())} '
      f'| studies with 0 labels: {(k==0).sum():,} ({(k==0).mean():.2%})')

## C9 · [YÊU CẦU 2] Phân phối 13 nhãn

Single measure (prevalence) across 13 categories → **horizontal bars, sorted, one
hue**. Not 13 colours: colour would just restate the bar length, and a categorical
palette implies identity that isn't there. The 2% line is §C's rarity threshold —
below it AUROC is inflated and AUPRC is the metric that matters.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

HUE, INK, MUTED, GRID = '#2a78d6', '#0b0b0b', '#52514e', '#e6e6e3'
RARE = 0.02

prev = (cohort[LABEL_COLS].mean().sort_values(ascending=True))
cnt  = cohort[LABEL_COLS].sum().reindex(prev.index)
disp = [c.replace('_', ' ') for c in prev.index]          # BQ cols carry underscores

fig, ax = plt.subplots(figsize=(9, 5.6), dpi=140)
ax.barh(disp, prev.values, height=0.72, color=HUE, zorder=3)
ax.axvline(RARE, color=MUTED, lw=1, ls='--', zorder=4)
ax.text(RARE, len(prev)-0.35, '  2% — rare-label threshold', va='center',
        ha='left', fontsize=8, color=MUTED)

for y,(p,c) in enumerate(zip(prev.values, cnt.values)):
    ax.text(p + prev.max()*0.012, y, f'{p:.1%}  (n={int(c):,})',
            va='center', fontsize=8.5, color=INK)

ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0, decimals=0))
ax.set_xlim(0, prev.max()*1.26)
ax.set_xlabel('Prevalence (share of studies)', fontsize=9, color=MUTED)
ax.set_title(f'CheXpert label prevalence — {len(LABEL_COLS)} labels, '
             f'{len(cohort):,} studies (U-{U_POLICY})', fontsize=11, color=INK, pad=12)
ax.grid(axis='x', color=GRID, lw=0.8, zorder=0); ax.set_axisbelow(True)
for s in ('top','right','left'): ax.spines[s].set_visible(False)
ax.spines['bottom'].set_color(GRID)
ax.tick_params(axis='y', length=0, labelsize=9, colors=INK)
ax.tick_params(axis='x', length=0, labelsize=8, colors=MUTED)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR,'label_prevalence.png'), dpi=160, bbox_inches='tight')
plt.show()

# head/med/tail split by TRAIN frequency (§C metrics breakdown)
tr = cohort[cohort.split=='train']
pt = tr[LABEL_COLS].mean().sort_values(ascending=False)
tier = pd.cut(pt, [-1, 0.02, 0.10, 1.0], labels=['tail (<2%)','medium (2-10%)','head (>10%)'])
print('\nhead/medium/tail by TRAIN prevalence:')
print(pd.DataFrame({'train_prev':pt.round(4),'tier':tier}).to_string())

## C10 · [YÊU CẦU 3] Kiểm tra missing data

Critical distinction (§A6): a value being absent is **not** automatically
missingness. `cur_hadm = NULL` means *outpatient* — a real category, not a hole.
Reporting them as "missing" is what leads to dropping them.

In [ ]:
print('='*64); print('MISSING DATA — cohort_base'); print('='*64)
miss = pd.DataFrame({
    'n_null':  cohort[base_cols].isna().sum(),
    'pct_null':(cohort[base_cols].isna().mean()*100).round(2),
    'dtype':   cohort[base_cols].dtypes.astype(str),
    'n_unique':cohort[base_cols].nunique(),
})
print(miss.to_string())

print('\n--- interpretation: NULL that is NOT missingness ---')
for c, meaning in [('cur_hadm','no inpatient admission at index_time -> OUTPATIENT/ED'),
                   ('cur_edstay','no ED stay at index_time -> inpatient/outpatient')]:
    print(f'  {c:<12} {cohort[c].isna().mean():6.2%} null  ->  {meaning}')
print('  => these are CATEGORIES (encoded in `setting`), not holes to impute.')

print('\n--- true integrity checks (all must be 0) ---')
chk = {
  'study_id null':          int(cohort.study_id.isna().sum()),
  'study_id duplicated':    int(cohort.study_id.duplicated().sum()),
  'subject_id null':        int(cohort.subject_id.isna().sum()),
  'index_time null':        int(cohort.index_time.isna().sum()),
  'split null':             int(cohort.split.isna().sum()),
  'setting null':           int(cohort.setting.isna().sum()),
  'label null':             int(cohort[LABEL_COLS].isna().sum().sum()),
  'label not in {0,1}':     int((~cohort[LABEL_COLS].isin([0,1])).sum().sum()),
  'n_images < 1':           int((cohort.n_images < 1).sum()),
}
for k,v in chk.items():
    print(f'  {"OK " if v==0 else "!! "} {k:<24} {v}')
assert all(v==0 for v in chk.values()), f'integrity FAILED: {chk}'

## C11 · §E1 acceptance tests available at this stage

The micro/leakage asserts (`frac_studies_with_future_organism == 0.0`,
`frac_studies_with_no_micro ∈ [0.40,0.60]`, `org_itemid` NULL ≈ 0.59) belong to
`02_ehr_features.ipynb` — they need the micro join. Everything cohort-level runs
here.

In [ ]:
assert cohort.groupby('study_id').size().max() == 1, 'study_id not unique'
print('OK  one row per study_id')

for a in ['train','val','test']:
    for b in ['train','val','test']:
        if a < b: assert not (S[a] & S[b])
print('OK  train/val/test subject_id sets disjoint')

assert len(LABEL_COLS) == 13 and 'Support Devices' not in LABEL_COLS
print(f'OK  {len(LABEL_COLS)} labels, Support Devices dropped')

assert cohort['index_time'].notna().all()
print('OK  every study has a valid index_time')

s_out = (cohort.setting=='outpatient').mean()
print(f'\nsetting: outpatient {s_out:.1%} | ed {(cohort.setting=="ed").mean():.1%} '
      f'| inpatient {(cohort.setting=="inpatient").mean():.1%}')
print('   -> §A4 predicts Tier-1 (current-encounter micro) will be rare (~11%);')
print('      that is explained by this distribution, not by missing data.')

print('\nCOHORT STAGE PASSED — next: 02_ehr_features.ipynb (A3-A7 + E1 leakage asserts)')